# Ch 23. KV 캐시 — 해설

> 이 노트북은 다섯 연습문제의 재현 가능한 해설을 제공한다. 모든 출력은 비워 두었으며 코드 셀은 위에서 아래로 독립적인 CPU 환경에서 실행된다.


## 문제 1

**문제**: KV 캐시 on/off로 100토큰 생성 시간을 n=128, 512, 2048에서 비교하라.

### 해설

캐시가 없으면 각 decode 단계에서 길이 전체의 K,V를 다시 계산해 비용이 대략 $\sum_{t=n}^{n+99}t$에 비례한다. 캐시가 있으면 새 토큰만 투영해 100에 비례한다. 실제 시간은 하드웨어 의존이므로 아래는 정확한 작업량 대리값과 증가율을 제시한다.


In [ ]:
import time
import numpy as np

# Reduced autoregressive projection benchmark: recompute the prefix or append one cached row.
rng=np.random.default_rng(2301); width=64; generated=100; W=rng.normal(size=(width,width)); results={}
for initial in (128,512,2048):
    tokens=rng.normal(size=(initial+generated,width))
    start=time.perf_counter();
    for step in range(generated): tokens[:initial+step+1]@W
    no_cache=time.perf_counter()-start
    start=time.perf_counter(); cache=tokens[:initial]@W
    for step in range(generated): cache=np.vstack((cache,tokens[initial+step:initial+step+1]@W))
    cached=time.perf_counter()-start
    results[initial]={"no_cache_ms":1000*no_cache,"cache_ms":1000*cached,"speedup":no_cache/cached}
assert all(r["speedup"]>1 for r in results.values())
print({n:{k:round(v,3) for k,v in r.items()} for n,r in results.items()})


## 문제 2

**문제**: LLaMA-7B에서 컨텍스트 32K의 KV 캐시 메모리를 계산하라.

### 해설

LLaMA-7B는 32층, 32 KV 헤드, 헤드 차원 128이다. FP16에서 메모리는 $2\times32\times32\times128\times32768\times2$ bytes $=16$ GiB다. 첫 2는 K와 V, 마지막 2는 원소당 바이트다.


In [ ]:
layers,kv_heads,head_dim,context,batch,bytes_per_value=32,32,128,32768,1,2
bytes_used=2*layers*kv_heads*head_dim*context*batch*bytes_per_value
gib=bytes_used/1024**3
assert bytes_used==17_179_869_184 and gib==16.0
print({"layers":layers,"kv_heads":kv_heads,"head_dim":head_dim,"context":context,"batch":batch,
       "dtype":"fp16","KV_cache_bytes":bytes_used,"KV_cache_GiB":gib})


## 문제 3

**문제**: GQA가 KV 캐시 메모리를 얼마나 절약하는지 LLaMA-7B(32 헤드) vs LLaMA-70B(8 KV 헤드)로 비교하라.

### 해설

이름이 지정된 두 모델 전체를 비교할 때는 층 수도 포함해야 한다. 7B는 $32\times32=1024$ layer-heads, 70B는 $80\times8=640$ layer-heads이므로 같은 컨텍스트·헤드 차원·dtype에서 70B의 총 KV 캐시는 7B의 62.5%, 즉 37.5% 절약이다. 별도로 80층 아키텍처를 고정하고 KV 헤드만 32개에서 8개로 바꾸는 가상 비교에서는 75%를 절약한다.


In [ ]:
# Named-model total includes each architecture's layer count; fixed-architecture isolates head sharing.
batch,context,head_dim,bytes_per_value=1,32768,128,2
def kv_bytes(layers,heads): return 2*batch*layers*heads*context*head_dim*bytes_per_value
llama_7b=kv_bytes(32,32); llama_70b=kv_bytes(80,8)
named_saving=1-llama_70b/llama_7b
fixed_mha=kv_bytes(80,32); fixed_gqa=kv_bytes(80,8); fixed_saving=1-fixed_gqa/fixed_mha
assert named_saving==0.375 and fixed_saving==0.75
print({"named_models":{"LLaMA-7B_GiB":llama_7b/1024**3,"LLaMA-70B_GiB":llama_70b/1024**3,
       "70B_vs_7B_saving":named_saving},"fixed_80_layer_architecture":{"32_to_8_KV_heads_saving":fixed_saving}})


## 문제 4

**문제**: Continuous Batching의 처리량 향상을 시뮬레이션으로 보이라.

### 해설

정적 배치는 가장 긴 요청이 끝날 때까지 빈 슬롯을 유지한다. continuous batching은 완료 즉시 새 요청을 넣어 활성 슬롯 시간을 늘린다. 아래 이산 이벤트 시뮬레이션은 동일 요청 길이에서 총 완료 시간과 처리량을 결정적으로 비교한다.


In [ ]:
import heapq

lengths=[9,2,7,3,8,4,6,5,2,9,3,7]; slots=3
static_time=sum(max(lengths[i:i+slots]) for i in range(0,len(lengths),slots))
workers=[0]*slots
for length in lengths:
    earliest=heapq.heappop(workers); heapq.heappush(workers,earliest+length)
continuous_time=max(workers)
tokens=sum(lengths); static_throughput=tokens/static_time; continuous_throughput=tokens/continuous_time
assert continuous_throughput>static_throughput
print({"requests":lengths,"static_ticks":static_time,"continuous_ticks":continuous_time,
       "static_tokens_per_tick":round(static_throughput,3),"continuous_tokens_per_tick":round(continuous_throughput,3),
       "throughput_gain":round(continuous_throughput/static_throughput,3)})


## 문제 5

**문제**: PagedAttention이 메모리 단편화를 어떻게 해결하는지 설명하라.

### 해설

PagedAttention은 논리적 연속 KV를 고정 크기 물리 블록에 나누고 페이지 테이블로 매핑한다. 요청별 최대 길이의 연속 공간을 미리 잡지 않아 외부 단편화를 줄이고, 마지막 블록의 최대 한 블록 미만만 내부 낭비가 된다. 블록 공유는 prefix 재사용도 가능하게 한다.


In [ ]:
import math

requests=[17,33,65,7]; block=16
contiguous_reserved=[64,64,128,32]
paged_reserved=[math.ceil(n/block)*block for n in requests]
contiguous_waste=sum(contiguous_reserved)-sum(requests); paged_waste=sum(paged_reserved)-sum(requests)
page_table={request:list(range(sum(paged_reserved[:i])//block,sum(paged_reserved[:i+1])//block)) for i,request in enumerate(requests)}
assert paged_waste < contiguous_waste and all(waste < block for waste in [r-n for r,n in zip(paged_reserved,requests)])
print({"request_tokens":requests,"contiguous_waste":contiguous_waste,"paged_waste":paged_waste,"page_table":page_table})
